# Run the reliability experiment on DKT

## Imports and clone repo

In [ ]:
!git clone https://github.com/aarushban-12/knowledge-tracing-collection-pytorch.git

%cd /content/knowledge-tracing-collection-pytorch

!git remote -v

import os
import random
import pickle
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

from sklearn.metrics import roc_auc_score, brier_score_loss

from models.dkt import DKT
from models.utils import match_seq_len, collate_fn


# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)
print("CUDA available:", torch.cuda.is_available())

Cloning into 'knowledge-tracing-collection-pytorch'...
remote: Enumerating objects: 488, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 488 (delta 22), reused 4 (delta 0), pack-reused 441 (from 1)
Receiving objects: 100% (488/488), 1.11 MiB | 10.27 MiB/s, done.
Resolving deltas: 100% (252/252), done.
/content/knowledge-tracing-collection-pytorch
origin	https://github.com/aarushban-12/knowledge-tracing-collection-pytorch.git (fetch)
origin	https://github.com/aarushban-12/knowledge-tracing-collection-pytorch.git (push)
Seed: 42
CUDA available: True


## Import datasets

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

RELIABILITY_DIR = (
    "/content/drive/MyDrive/education-ml-research/"
    "ASSISTments2009/reliability_experiment"
)

TRAIN_PATH = os.path.join(
    RELIABILITY_DIR,
    "train.csv"
)

TEST_PATHS = {
    "Q1": os.path.join(RELIABILITY_DIR, "q1_test.csv"),
    "Q2": os.path.join(RELIABILITY_DIR, "q2_test.csv"),
    "Q3": os.path.join(RELIABILITY_DIR, "q3_test.csv"),
    "Q4": os.path.join(RELIABILITY_DIR, "q4_test.csv"),
}

print("Training file exists:", os.path.exists(TRAIN_PATH))

for q, path in TEST_PATHS.items():
    print(q, "exists:", os.path.exists(path))

train_df = pd.read_csv(
    TRAIN_PATH,
    low_memory=False
)

test_dfs = {
    q: pd.read_csv(path, low_memory=False)
    for q, path in TEST_PATHS.items()
}

print("Training shape:", train_df.shape)

for q, df_q in test_dfs.items():
    print(f"{q} shape:", df_q.shape)

print(
    "Training students:",
    train_df["user_id"].nunique()
)

print("Testing students:")

for q, df_q in test_dfs.items():
    print(
        f"{q} students:",
        df_q["user_id"].nunique()
    )

Mounted at /content/drive
Training file exists: True
Q1 exists: True
Q2 exists: True
Q3 exists: True
Q4 exists: True
Training shape: (426580, 30)
Q1 shape: (11444, 30)
Q2 shape: (23524, 30)
Q3 shape: (30184, 30)
Q4 shape: (33802, 30)
Training students: 3371
Testing students:
Q1 students: 212
Q2 students: 211
Q3 students: 212
Q4 students: 211


## Create skill mapping so each of the skills have same id across all datasets

In [ ]:
train_df = train_df.dropna(
    subset=["skill_name"]
).copy()

for q in test_dfs:
    test_dfs[q] = test_dfs[q].dropna(
        subset=["skill_name"]
    ).copy()

q_list = np.unique(
    train_df["skill_name"].values
)

q2idx = {
    q: idx
    for idx, q in enumerate(q_list)
}

num_q = len(q_list)

print("Number of skills:", num_q)

for q, df_q in test_dfs.items():

    unseen = (
        set(df_q["skill_name"].unique())
        - set(q2idx.keys())
    )

    print(
        f"{q}: unseen skills = {len(unseen)}"
    )

    if len(unseen) > 0:
        print(
            "WARNING: unseen skills:",
            list(unseen)[:10]
        )

Number of skills: 110
Q1: unseen skills = 0
Q2: unseen skills = 0
Q3: unseen skills = 0
Q4: unseen skills = 0


## Create custom dataset using shared mapping (This resembles the ASSIST2009 dataloader from the hcnoh repo)

In [ ]:
class ReliabilityASSIST2009(Dataset):

    def __init__(
        self,
        df,
        q2idx,
        seq_len=100
    ):

        super().__init__()

        self.dataset_dir = None
        self.q2idx = q2idx

        self.q_list = np.array(
            list(q2idx.keys())
        )

        self.num_q = len(q2idx)

        df = df.copy()

        df = df.dropna(
            subset=["skill_name"]
        )

        df = df.drop_duplicates(
            subset=["order_id", "skill_name"]
        )

        df = df.sort_values(
            by=["order_id"]
        )

        self.u_list = np.unique(
            df["user_id"].values
        )

        self.q_seqs = []
        self.r_seqs = []

        for u in self.u_list:

            df_u = df[
                df["user_id"] == u
            ]

            q_seq = np.array([
                q2idx[q]
                for q in df_u["skill_name"]
            ])

            r_seq = df_u["correct"].values.astype(
                np.float32
            )

            self.q_seqs.append(q_seq)
            self.r_seqs.append(r_seq)

        if seq_len:

            self.q_seqs, self.r_seqs = match_seq_len(
                self.q_seqs,
                self.r_seqs,
                seq_len
            )

        self.len = len(self.q_seqs)

    def __getitem__(self, index):

        return (
            self.q_seqs[index],
            self.r_seqs[index]
        )

    def __len__(self):

        return self.len

SEQ_LEN = 100

train_dataset = ReliabilityASSIST2009(
    train_df,
    q2idx,
    seq_len=SEQ_LEN
)

print(
    "Training sequences:",
    len(train_dataset)
)

print(
    "Number of skills:",
    train_dataset.num_q
)

test_datasets = {}

for q, df_q in test_dfs.items():

    test_datasets[q] = ReliabilityASSIST2009(
        df_q,
        q2idx,
        seq_len=SEQ_LEN
    )

    print(
        q,
        "sequences:",
        len(test_datasets[q])
    )

Training sequences: 5024
Number of skills: 110
Q1 sequences: 243
Q2 sequences: 300
Q3 sequences: 366
Q4 sequences: 271


## Create validation split in order to see and select best AUC

In [ ]:
from torch.utils.data import random_split

torch.set_default_device("cpu")

VAL_RATIO = 0.10

train_size = int(
    len(train_dataset) * (1 - VAL_RATIO)
)

val_size = (
    len(train_dataset) - train_size
)

generator = torch.Generator(
    device="cpu"
)

generator.manual_seed(SEED)

train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=generator
)

print(
    "Training sequences:",
    len(train_subset)
)

print(
    "Validation sequences:",
    len(val_subset)
)

Training sequences: 4521
Validation sequences: 503


## Create dataloaders

In [ ]:
BATCH_SIZE = 128

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_subset,
    batch_size=len(val_subset),
    shuffle=False,
    collate_fn=collate_fn
)

test_loaders = {}

for q, dataset in test_datasets.items():

    test_loaders[q] = DataLoader(
        dataset,
        batch_size=len(dataset),
        shuffle=False,
        collate_fn=collate_fn
    )

print("DataLoaders created.")

DataLoaders created.


## Create the DKT model

In [ ]:
BATCH_SIZE = 128
NUM_EPOCHS = 100
LEARNING_RATE = 0.001
SEQ_LEN = 100

EMBED_DIM = 100
HIDDEN_SIZE = 100
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = DKT(
    num_q=num_q,
    emb_size=EMBED_DIM,
    hidden_size=HIDDEN_SIZE
).to(device)

optimizer = Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print("Device:", device)
print("Model:", model)

Device: cuda
Model: DKT(
  (interaction_emb): Embedding(220, 100)
  (lstm_layer): LSTM(100, 100, batch_first=True)
  (out_layer): Linear(in_features=100, out_features=110, bias=True)
  (dropout_layer): Dropout(p=0.5, inplace=False)
)


## Evaluate function

In [ ]:
def evaluate_dkt(model, loader, device):
    model.eval()

    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for data in loader:
            q, r, qshft, rshft, m = data

            q = q.long().to(device)
            r = r.long().to(device)
            qshft = qshft.long().to(device)
            rshft = rshft.float().to(device)
            m = m.to(device)

            # DKT forward pass
            p = model(q, r)

            # Select prediction for the actual next skill
            p = p.gather(
                2,
                qshft.unsqueeze(-1)
            ).squeeze(-1)

            # Keep only valid sequence positions
            p = torch.masked_select(p, m)
            t = torch.masked_select(rshft, m)

            all_predictions.append(
                p.detach().cpu().numpy()
            )

            all_targets.append(
                t.detach().cpu().numpy()
            )

    predictions = np.concatenate(all_predictions)
    targets = np.concatenate(all_targets)

    # AUC
    auc = roc_auc_score(
        targets,
        predictions
    )

    # Brier score
    brier = brier_score_loss(
        targets,
        predictions
    )

    return auc, brier

## Training function

In [ ]:
def train_dkt(
    model,
    train_loader,
    val_loader,
    optimizer,
    num_epochs,
    device,
    checkpoint_path
):

    best_auc = -np.inf
    best_epoch = None

    auc_history = []
    loss_history = []

    for epoch in range(1, num_epochs + 1):

        model.train()

        epoch_losses = []

        for data in train_loader:

            q, r, qshft, rshft, m = data

            q = q.long().to(device)
            r = r.long().to(device)
            qshft = qshft.long().to(device)
            rshft = rshft.float().to(device)
            m = m.to(device)

            # Forward pass
            p = model(
                q,
                r
            )

            # Select prediction for the actual next skill
            p = p.gather(
                2,
                qshft.unsqueeze(-1)
            ).squeeze(-1)

            # Select valid positions
            p = torch.masked_select(
                p,
                m
            )

            t = torch.masked_select(
                rshft,
                m
            )

            # Binary cross entropy
            loss = nn.functional.binary_cross_entropy(
                p,
                t
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            epoch_losses.append(
                loss.detach().cpu().item()
            )

        mean_loss = np.mean(epoch_losses)

        # Validation AUC
        val_auc, val_brier = evaluate_dkt(
            model,
            val_loader,
            device
        )

        auc_history.append(val_auc)
        loss_history.append(mean_loss)

        print(
            f"Epoch {epoch:03d} | "
            f"AUC: {val_auc:.6f} | "
            f"Loss: {mean_loss:.6f}"
        )

        # Save best checkpoint
        if val_auc > best_auc:

            best_auc = val_auc
            best_epoch = epoch

            torch.save(
                model.state_dict(),
                checkpoint_path
            )

            print(
                f"  → New best model "
                f"(AUC={best_auc:.6f})"
            )

    return (
        auc_history,
        loss_history,
        best_auc,
        best_epoch
    )

## Create best checkpoint directory

In [ ]:
CHECKPOINT_DIR = os.path.join(
    RELIABILITY_DIR,
    "dkt_reliability"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "dkt_best.pt"
)

print(
    "Checkpoint path:",
    CHECKPOINT_PATH
)

Checkpoint path: /content/drive/MyDrive/education-ml-research/ASSISTments2009/reliability_experiment/dkt_reliability/dkt_best.pt


## Train DKT

In [ ]:
auc_history, loss_history, best_val_auc, best_epoch = train_dkt(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    num_epochs=NUM_EPOCHS,
    device=device,
    checkpoint_path=CHECKPOINT_PATH
)

print("\nTraining complete.")
print("Best validation AUC:", best_val_auc)
print("Best epoch:", best_epoch)

/content/knowledge-tracing-collection-pytorch/models/utils.py:91: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  q_seqs.append(FloatTensor(q_seq[:-1]))


Epoch 001 | AUC: 0.742339 | Loss: 0.647653
  → New best model (AUC=0.742339)
Epoch 002 | AUC: 0.774901 | Loss: 0.615713
  → New best model (AUC=0.774901)
Epoch 003 | AUC: 0.791460 | Loss: 0.602796
  → New best model (AUC=0.791460)
Epoch 004 | AUC: 0.798666 | Loss: 0.596544
  → New best model (AUC=0.798666)
Epoch 005 | AUC: 0.802977 | Loss: 0.592431
  → New best model (AUC=0.802977)
Epoch 006 | AUC: 0.806295 | Loss: 0.589354
  → New best model (AUC=0.806295)
Epoch 007 | AUC: 0.808504 | Loss: 0.587475
  → New best model (AUC=0.808504)
Epoch 008 | AUC: 0.810710 | Loss: 0.586118
  → New best model (AUC=0.810710)
Epoch 009 | AUC: 0.812192 | Loss: 0.582792
  → New best model (AUC=0.812192)
Epoch 010 | AUC: 0.813554 | Loss: 0.582356
  → New best model (AUC=0.813554)
Epoch 011 | AUC: 0.813646 | Loss: 0.580263
  → New best model (AUC=0.813646)
Epoch 012 | AUC: 0.814848 | Loss: 0.578553
  → New best model (AUC=0.814848)
Epoch 013 | AUC: 0.815490 | Loss: 0.578065
  → New best model (AUC=0.815490)

## Load best DKT checkpoint

In [ ]:
model.load_state_dict(
    torch.load(
        CHECKPOINT_PATH,
        map_location=device
    )
)

model.to(device)
model.eval()

print("Best DKT checkpoint loaded.")

Best DKT checkpoint loaded.


## Test model on quartiles

In [ ]:
quartile_results = {}

for q in ["Q1", "Q2", "Q3", "Q4"]:

    auc, brier = evaluate_dkt(
        model,
        test_loaders[q],
        device
    )

    quartile_results[q] = {
        "auc": auc,
        "brier": brier
    }

    print(
        f"{q} AUC: {auc:.6f} | "
        f"Brier Score: {brier:.6f}"
    )

Q1 AUC: 0.817947 | Brier Score: 0.174842
Q2 AUC: 0.757903 | Brier Score: 0.195070
Q3 AUC: 0.761489 | Brier Score: 0.162813
Q4 AUC: 0.765060 | Brier Score: 0.126181


## Create table and save results

In [ ]:
results_df = pd.DataFrame({
    "Ability Quartile": ["Q1", "Q2", "Q3", "Q4"],
    "DKT AUC": [
        quartile_results[q]["auc"]
        for q in ["Q1", "Q2", "Q3", "Q4"]
    ],
    "DKT Brier Score": [
        quartile_results[q]["brier"]
        for q in ["Q1", "Q2", "Q3", "Q4"]
    ]
})

results_df
results_df

RESULTS_PATH = os.path.join(
    CHECKPOINT_DIR,
    "dkt_reliability_results.csv"
)

results_df.to_csv(
    RESULTS_PATH,
    index=False
)

print("Saved results to:", RESULTS_PATH)

Saved results to: /content/drive/MyDrive/education-ml-research/ASSISTments2009/reliability_experiment/dkt_reliability/dkt_reliability_results.csv


## Results

The DKT model was trained once using the predefined training dataset and then evaluated separately on the four existing student-ability quartile test sets. The model used an embedding size of 100, hidden size of 100, batch size of 128, sequence length of 100, learning rate of 0.001, and the Adam optimizer. The model was trained for up to 100 epochs, with the best model selected based on validation AUC.

The DKT model achieved a best validation AUC of 0.8169 at epoch 18. On the held-out ability quartile test sets, DKT achieved an AUC of 0.8179 for Q1, 0.7579 for Q2, 0.7615 for Q3, and 0.7651 for Q4. The highest predictive performance occurred for Q1 students, while performance decreased substantially for Q2 and then remained relatively stable across Q2–Q4. The difference between the highest and lowest quartile AUCs was approximately 0.0600, indicating that DKT's ability to distinguish between correct and incorrect responses varied across student-ability groups.

Brier scores were also calculated to evaluate the accuracy of DKT's predicted probabilities. The Brier score was 0.1748 for Q1, 0.1951 for Q2, 0.1628 for Q3, and 0.1262 for Q4. Unlike the AUC results, the Brier scores generally improved as student ability increased, with Q4 producing the lowest Brier score (0.1262). Q2 had the highest Brier score (0.1951), indicating the greatest probabilistic prediction error among the four groups.

The AUC and Brier score results therefore show different patterns across ability groups. AUC was highest for Q1 and decreased for the remaining groups, whereas Brier scores were generally lower for the higher-ability groups, with Q4 having the most accurate predicted probabilities. This distinction suggests that DKT's ability to rank correct versus incorrect responses and the accuracy of its predicted probabilities were affected differently by student ability.

The difference in AUC between the best- and worst-performing quartiles was approximately 0.0600, while the difference in Brier score was approximately 0.0689. This indicates that student ability was associated with differences not only in DKT's discriminative performance but also in the accuracy of its probability estimates.

Overall, these findings provide evidence that student ability is associated with differences in DKT predictive performance. However, the relatively modest AUC variation across quartiles suggests that DKT was comparatively stable in its ability to distinguish correct from incorrect responses. The Brier scores provide additional evidence that the accuracy of DKT's probabilistic predictions varied across ability groups. These results can be compared with SAKT, BKT, and Logistic Regression to determine whether model architecture influences the reliability of knowledge tracing predictions across different student-ability groups.